In [3]:
import requests
import pandas as pd
from datetime import datetime, timedelta

def get_itcen_group_bids():
    """아이티센 그룹 맞춤형 공고 및 제안요청서 다운로드 링크 추출 함수"""
    
    # 1. API 기본 설정
    SERVICE_KEY = "18dbc10d66b4f8764dcc5ada60ebe8550b7232c4a4883480afd6c24d8c5aed78"
    BASE_URL = "https://apis.data.go.kr/1230000/ad/BidPublicInfoService/getBidPblancListInfoServc"

    # 조회 기간 설정 (기본: 최근 7일)
    # 조회 기간 바꾸면서 api 갱신하며 사용
    today = datetime.now()
    seven_days_ago = today - timedelta(days=7)
    bgn_dt = seven_days_ago.strftime("%Y%m%d0000")
    end_dt = today.strftime("%Y%m%d2359")

    # 2. 그룹사별 타겟 분류 코드 분리
    CLOIT_CODES = ['81111513', '81111595', '81111596', '81111594', '81111599', '81111598', '81112002', '80101507', '80101698']
    NTEC_CODES = ['81111799', '81111801', '81111809', '81111708', '81151699']
    GLOBAL_CODES = ['81111899', '81112399', '81112299', '81111811', '81112199', '80141619']
    ALL_TARGET_CODES = CLOIT_CODES + NTEC_CODES + GLOBAL_CODES

    result_count = 100
    url = (
        f"{BASE_URL}?serviceKey={SERVICE_KEY}"
        f"&pageNo=1&numOfRows={result_count}&type=json&inqryDiv=1"
        f"&inqryBgnDt={bgn_dt}&inqryEndDt={end_dt}"
    )

    print("⏳ [로딩 중] 조달청 API에서 공고 데이터를 수집 및 분석하고 있습니다. 잠시만 기다려주세요...\n")

    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        data = response.json()

        items = data.get('response', {}).get('body', {}).get('items', [])
        filtered_results = []
        
        # 추출 타겟 키워드 리스트
        target_keywords = ["제안", "과업", "입찰"]
        
        # 3. 공고 목록 순회 및 하드 필터링
        for item in items:
            clsfc_no = item.get('pubPrcrmntClsfcNo')
            
            if clsfc_no in ALL_TARGET_CODES:
                
                if clsfc_no in CLOIT_CODES:
                    target_company = "☁️ 클로잇 (CloIT)"
                elif clsfc_no in NTEC_CODES:
                    target_company = "🌐 엔텍 (NTEC)"
                else:
                    target_company = "🏢 글로벌 (Global)"
                
                bid_info = {
                    "targetCompany": target_company,
                    "bidNtceNo": item.get('bidNtceNo'),
                    "bidNtceNm": item.get('bidNtceNm'),
                    "ntceInsttNm": item.get('ntceInsttNm'),
                    "pubPrcrmntClsfcNm": item.get('pubPrcrmntClsfcNm'),
                    "bidClseDt": item.get('bidClseDt'),
                    "asignBdgtAmt": item.get('asignBdgtAmt'),
                    "bidNtceDtlUrl": item.get('bidNtceDtlUrl'), 
                    "targetFiles": [] 
                }
                
                # 4. 첨부파일(1~10) 텍스트 검사 및 링크 추출 (리스트 순회)
                for i in range(1, result_count+1):
                    file_nm = item.get(f"ntceSpecFileNm{i}", "")
                    file_url = item.get(f"ntceSpecDocUrl{i}", "")
                    
                    if file_nm and any(keyword in file_nm for keyword in target_keywords):
                        bid_info["targetFiles"].append({
                            "fileName": file_nm,
                            "downloadUrl": file_url
                        })
                
                filtered_results.append(bid_info)
                
        return filtered_results

    except Exception as e:
        print(f"❌ 데이터 수집 에러 발생: {str(e)}")
        return []

def save_to_excel(results):
    """추출된 리스트를 바탕으로 엑셀 파일 생성"""
    if not results:
        return
    
    excel_data = []
    
    for r in results:
        # 예산 텍스트 포맷팅
        budget = f"{int(r['asignBdgtAmt']):,}원" if r['asignBdgtAmt'] else "미정"
        
        # 여러 개의 첨부파일과 링크를 하나의 셀에 줄바꿈으로 이어서 저장
        files_text = ""
        for idx, f in enumerate(r['targetFiles'], 1):
            files_text += f"{idx}. {f['fileName']}\n🔗 {f['downloadUrl']}\n\n"
            
        if not files_text:
            files_text = "타겟 첨부파일 없음"
            
        excel_data.append({
            "타겟 그룹사": r['targetCompany'],
            "공고명": r['bidNtceNm'],
            "세부분류명": r['pubPrcrmntClsfcNm'],
            "수요기관": r['ntceInsttNm'],
            "예산": budget,
            "마감일시": r['bidClseDt'],
            "공고 상세링크": r['bidNtceDtlUrl'],
            "첨부파일 다운로드": files_text.strip()
        })
        
    df = pd.DataFrame(excel_data)
    
    file_name = f"아이티센_그룹_맞춤형_공고_{datetime.now().strftime('%Y%m%d')}.csv"
    
    df.to_csv(file_name, index=False, encoding='utf-8-sig')
    print(f"\n💾 CSV 저장 완료: [{file_name}] 파일이 현재 폴더에 생성됨.")


if __name__ == "__main__":
    results = get_itcen_group_bids()
    
    if not results:
        print("조건에 맞는 타겟 공고가 없거나 조회에 실패함.")
    else:
        print(f"✅ 총 {len(results)}건의 맞춤형 공고가 검색됨.\n")
        print("=" * 60)
        
        results.sort(key=lambda x: x['targetCompany'])
        current_company = ""
        
        for r in results:
            if current_company != r['targetCompany']:
                current_company = r['targetCompany']
                print(f"\n{current_company} 타겟 공고 목록")
                print("-" * 60)
            
            budget = f"{int(r['asignBdgtAmt']):,}" if r['asignBdgtAmt'] else "미정"
            
            print(f"📌 [공고명] {r['bidNtceNm']}")
            print(f"   - 분류명 : {r['pubPrcrmntClsfcNm']}")
            print(f"   - 기관명 : {r['ntceInsttNm']}")
            print(f"   - 예  산 : {budget}원")
            print(f"   - 마감일 : {r['bidClseDt']}")
            print(f"   - 🔗 공고 원본 이동: {r['bidNtceDtlUrl']}")
            
            if r['targetFiles']:
                print("   - 📎 첨부파일 (클릭 시 다운로드):")
                for f in r['targetFiles']:
                    print(f"     └─ [{f['fileName']}]")
                    print(f"        {f['downloadUrl']}")
            else:
                print("   - 📎 타겟 첨부파일(제안/과업/입찰) 없음")
                
            print("=" * 60)
            
        # 콘솔 출력 후 엑셀 저장 함수 실행
        save_to_excel(results)

⏳ [로딩 중] 조달청 API에서 공고 데이터를 수집 및 분석하고 있습니다. 잠시만 기다려주세요...

✅ 총 10건의 맞춤형 공고가 검색됨.


☁️ 클로잇 (CloIT) 타겟 공고 목록
------------------------------------------------------------
📌 [공고명] DW/BI 입금지표 및 후원지표 개발
   - 분류명 : 정보시스템개발서비스
   - 기관명 : 사회복지법인 세이브더칠드런코리아
   - 예  산 : 18,270,000원
   - 마감일 : 2026-06-17 12:00:00
   - 🔗 공고 원본 이동: https://www.g2b.go.kr/link/PNPE027_01/single/?bidPbancNo=R26BK01582510&bidPbancOrd=000
   - 📎 첨부파일 (클릭 시 다운로드):
     └─ [과업지시서_입금지표_및_후원지표_개발.docx]
        https://www.g2b.go.kr/pn/pnp/pnpe/UntyAtchFile/downloadFile.do?bidPbancNo=R26BK01582510&bidPbancOrd=000&fileType=&fileSeq=2&prcmBsneSeCd=03
📌 [공고명] Top-tier 연구소 홈페이지 구축
   - 분류명 : 정보시스템개발서비스
   - 기관명 : 조선대학교
   - 예  산 : 50,000,000원
   - 마감일 : 
   - 🔗 공고 원본 이동: https://www.g2b.go.kr/link/PNPE027_01/single/?bidPbancNo=R26BK01581420&bidPbancOrd=001
   - 📎 첨부파일 (클릭 시 다운로드):
     └─ [과업지시서(최종).hwp]
        https://www.g2b.go.kr/pn/pnp/pnpe/UntyAtchFile/downloadFile.do?bidPbancNo=R26BK01581420&bidPbancOrd=001&fileType=&fileSe